# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset information using metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the `@id` field to reference entities.

In [ ]:
# List record sets, fields, and columns with their @id
from pprint import pprint

record_sets = []
if hasattr(dataset.metadata, 'record_sets'):
    # Croissant >= 1.0
    record_sets = list(dataset.metadata.record_sets)
elif hasattr(dataset.metadata, 'recordSet'):
    record_sets = list(dataset.metadata.recordSet)

if not record_sets:
    print("No record sets found in the metadata. Aborting data overview.")
else:
    print('Available record sets:')
    for record_set in record_sets:
        if hasattr(record_set, '@id'):
            print(f"- Record set @id: {record_set['@id']} | name: {record_set.get('name', 'N/A')}")
            if 'field' in record_set:
                print('  Fields:')
                fields = record_set['field'] if isinstance(record_set['field'], list) else [record_set['field']]
                for field in fields:
                    if isinstance(field, dict):
                        print(f"    - Field @id: {field.get('@id', 'N/A')}, name: {field.get('name', 'N/A')}")
                        if 'column' in field:
                            columns = field['column'] if isinstance(field['column'], list) else [field['column']]
                            for column in columns:
                                if isinstance(column, dict):
                                    print(f"      - Column @id: {column.get('@id', 'N/A')}, name: {column.get('name', 'N/A')}")
                                else:
                                    print(f"      - Column @id: {column}")
                    else:
                        print(f"    - Field @id: {field}")
        else:
            print(record_set)

    # Save the first record set and its fields/columns for the following cells
    record_set_ids = []
    field_ids = []
    # Attempt to collect @id
    for record_set in record_sets:
        if isinstance(record_set, dict) and '@id' in record_set:
            record_set_ids.append(record_set['@id'])
            if 'field' in record_set:
                fs = record_set['field']
                if isinstance(fs, list):
                    for f in fs:
                        if isinstance(f, dict) and '@id' in f:
                            field_ids.append(f['@id'])
                elif isinstance(fs, dict) and '@id' in fs:
                    field_ids.append(fs['@id'])
    print('\nRecord sets IDs:', record_set_ids)
    print('Field IDs:', field_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets available by their @id
# Use IDs extracted from the previous cell (edit as necessary if more record sets are found).
record_set_ids = [
    # Example: Replace with the appropriate @id from the output above
    # 'cr:RecordSet/ev_experiment_data'
    # If dataset.metadata.record_sets is empty, you may need to consult the dataset documentation
]
if not record_set_ids:
    print("No record set IDs found in the data overview. Please fill in the correct @id.")
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet @id={record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record set {record_set_id} DataFrame shape: {df.shape}')
        print(f'Columns: {df.columns.tolist()}')
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# This is a template cell. Please set the variables to match a numeric field and group field (by @id or name from above).
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Set these to the actual values from previous cell (see columns printed)
record_set_id = None     # e.g. 'cr:RecordSet/ev_experiment_data'
numeric_field = None     # e.g. 'cr:Field/abundance'
group_field = None       # e.g. 'cr:Field/sample'

if record_set_id and numeric_field and record_set_id in dataframes and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    threshold = df[numeric_field].quantile(0.95) # example threshold: top 5%
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("Please set 'record_set_id', 'numeric_field', and (optionally) 'group_field' to valid values from earlier cell output.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization for a numeric field - adjust variables to match those set above
if record_set_id and numeric_field and record_set_id in dataframes and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field is available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Set the record_set_id, numeric_field, and group_field to see visualizations.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

> In this notebook, we demonstrated how to load, explore, and visualize the FAIR² (FAIR-Squared) Mass Spectrometry dataset using the `mlcroissant` library. With the use of `@id` fields throughout, you can reliably reference and process specific entities in the Croissant schema. For further analysis, fill in the specific record set/field/column `@id`s as appropriate for your research goals.